# ITM-390: AI-Based Threat Detection Tool

This notebook implements the full workflow from dataset download to deployment-ready prediction, matching the ITM-390 project prompt.

Sections:
1. Environment Setup and Dependency Imports
2. Data Loading and Schema Validation
3. Exploratory Analysis for Threat Signals
4. Preprocessing and Feature Engineering
5. Class Imbalance Handling
6. Baseline Model Training
7. Model Evaluation with Security-Focused Metrics
8. Decision Threshold Tuning
9. Model Explainability for Alerts
10. Batch and Streaming Inference Pipeline
11. Unit Tests for Detection Components
12. Persisting Artifacts and Reproducible Runs

## 1. Environment Setup and Dependency Imports

Install required packages if needed, then import dependencies and set reproducibility controls.

In [ ]:
# Uncomment if packages are missing in your kernel:
# %pip install kagglehub scikit-learn pandas numpy matplotlib seaborn imbalanced-learn joblib

import glob
import os
import warnings

import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
OUTPUT_DIR = "output"
ARTIFACTS_DIR = "artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print("Libraries loaded. Output directory ready:", OUTPUT_DIR)
print("Artifacts directory ready:", ARTIFACTS_DIR)

In [ ]:
def download_dataset():
    dataset_id = "sampadab17/network-intrusion-detection"
    try:
        path = kagglehub.dataset_download(dataset_id)
        print(f"Downloaded: {dataset_id}")
        return dataset_id, path
    except Exception as exc:
        raise RuntimeError(f"Could not download dataset. Error: {exc}")
def pick_labeled_csv(csv_files):
    candidate_names = {"label", "class", "attack", "attack_cat", "target"}

    for csv_path in csv_files:
        try:
            head = pd.read_csv(csv_path, nrows=200)
        except Exception:
            continue
        head.columns = [str(col).strip().lower() for col in head.columns]
        if any(col in candidate_names for col in head.columns):
            return csv_path

    for csv_path in csv_files:
        try:
            head = pd.read_csv(csv_path, nrows=200)
        except Exception:
            continue
        head.columns = [str(col).strip().lower() for col in head.columns]
        for col in head.columns:
            if any(token in col for token in ["label", "class", "attack", "target"]):
                return csv_path

    for csv_path in csv_files:
        try:
            head = pd.read_csv(csv_path, nrows=1000)
        except Exception:
            continue
        head.columns = [str(col).strip().lower() for col in head.columns]
        last_col = head.columns[-1]
        unique_count = head[last_col].nunique(dropna=False)
        if str(head[last_col].dtype) == "object" or unique_count <= 20:
            return csv_path

    return csv_files[0]


def detect_target_column(columns, df=None):
    candidates = ["label", "class", "attack", "attack_cat", "target"]
    for candidate in candidates:
        if candidate in columns:
            return candidate

    if df is not None:
        fallback = []
        for col in columns:
            unique_count = df[col].nunique(dropna=False)
            if str(df[col].dtype) == "object" or unique_count <= 20:
                fallback.append((col, unique_count))
        if fallback:
            fallback.sort(key=lambda x: x[1])
            return fallback[0][0]

    return columns[-1]


def encode_target(y_series):
    y_str = y_series.astype(str).str.strip().str.lower()
    return y_str.apply(lambda value: 0 if value in {"normal", "benign", "0"} else 1)


def get_models(train_size):
    svm_entry = (
        "LinearSVC",
        LinearSVC(random_state=RANDOM_STATE),
    ) if train_size > 100000 else (
        "SVM",
        SVC(kernel="rbf", random_state=RANDOM_STATE, probability=True),
    )

    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        svm_entry[0]: svm_entry[1],
    }


def predict_threat(input_dict, model, scaler, selected_feature_names, threshold=0.5):
    row = pd.DataFrame([input_dict]).reindex(columns=selected_feature_names, fill_value=0)
    row_scaled = scaler.transform(row)
    row_selected = pd.DataFrame(row_scaled, columns=selected_feature_names)

    if hasattr(model, "predict_proba"):
        threat_prob = float(model.predict_proba(row_selected)[0][1])
    else:
        score = float(model.decision_function(row_selected)[0])
        threat_prob = 1.0 / (1.0 + np.exp(-score))

    pred = int(threat_prob >= threshold)
    label = "THREAT DETECTED" if pred == 1 else "NORMAL"
    return {
        "label": label,
        "threat_probability": threat_prob,
        "threshold": threshold,
    }

## 2. Data Loading and Schema Validation

Download one of the approved datasets via kagglehub, discover CSV files dynamically, load data, normalize column names, and validate schema.

In [ ]:
dataset_name, dataset_path = download_dataset()

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("Found CSV files:", csv_files[:5])
if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {dataset_path}")

csv_path = pick_labeled_csv(csv_files)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)
df.columns = [str(col).strip().lower() for col in df.columns]

target_col = detect_target_column(df.columns.tolist(), df=df)

print("\nShape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDetected target column:", target_col)
print("\nRaw target counts:\n", df[target_col].value_counts(dropna=False).head(20))

## 3. Exploratory Analysis for Threat Signals

Inspect class balance and visualize threat-vs-normal distribution.

In [ ]:
y_raw = df[target_col].astype(str).str.strip()
value_counts = y_raw.value_counts()
print(value_counts)

# Class distribution plot required by deliverables.
plt.figure(figsize=(8, 5))
value_counts.plot(kind="bar", color="#3182bd")
plt.title("Raw Class Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=200)
plt.show()

## 4. Preprocessing and Feature Engineering

Replace infinities, handle missing values, remove constant columns, binarize target, encode categoricals, split data, scale features, and select top 15 features.

In [ ]:
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

constant_cols = [
    c for c in df_clean.columns
    if c != target_col and df_clean[c].nunique(dropna=False) <= 1
]
if constant_cols:
    df_clean = df_clean.drop(columns=constant_cols)
print("Dropped constant cols:", constant_cols)

y = encode_target(df_clean[target_col])
X = df_clean.drop(columns=[target_col])
X = pd.get_dummies(X, drop_first=False)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index,
)

# SelectKBest(chi2) requires non-negative values.
X_train_chi2 = X_train_scaled.copy()
min_val = X_train_chi2.min().min()
if min_val < 0:
    X_train_chi2 = X_train_chi2 - min_val

k = min(15, X_train_chi2.shape[1])
selector = SelectKBest(chi2, k=k)
selector.fit(X_train_chi2, y_train)
selected_features = X_train_scaled.columns[selector.get_support()].tolist()
selected_scores = selector.scores_[selector.get_support()]

feature_df = pd.DataFrame({"feature": selected_features, "score": selected_scores}).sort_values("score", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(feature_df["feature"], feature_df["score"], color="#2b8cbe")
plt.title("Top Features by SelectKBest (chi2)")
plt.xlabel("Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance.png"), dpi=200)
plt.show()

X_train_sel = X_train_scaled[selected_features]
X_test_sel = X_test_scaled[selected_features]

# Build deployment scaler aligned to selected features so model/scaler/features stay in sync.
selected_indices = [X_train_raw.columns.get_loc(col) for col in selected_features]
deploy_scaler = StandardScaler()
deploy_scaler.mean_ = scaler.mean_[selected_indices]
deploy_scaler.var_ = scaler.var_[selected_indices]
deploy_scaler.scale_ = scaler.scale_[selected_indices]
deploy_scaler.n_features_in_ = len(selected_features)
deploy_scaler.feature_names_in_ = np.array(selected_features, dtype=object)
deploy_scaler.n_samples_seen_ = scaler.n_samples_seen_

print("Selected features:", selected_features)

## 5. Class Imbalance Handling

Apply SMOTE when minority class share is below 20%.

In [ ]:
ratio_before = y_train.value_counts(normalize=True).sort_index()
print("Before balancing:")
print(ratio_before)

if ratio_before.min() < 0.20:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_sel, y_train)
    print("SMOTE applied.")
else:
    X_train_bal, y_train_bal = X_train_sel.copy(), y_train.copy()
    print("SMOTE not required.")

ratio_after = y_train_bal.value_counts(normalize=True).sort_index()
print("\nAfter balancing:")
print(ratio_after)

## 6. Baseline Model Training

Train all required models with consistent train/test data.

In [ ]:
models = get_models(len(X_train_bal))
results = []
reports = {}
conf_mats = {}
fitted = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    pred = model.predict(X_test_sel)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, pred),
        "PR-AUC": average_precision_score(y_test, pred),
    })
    reports[name] = classification_report(y_test, pred, digits=4)
    conf_mats[name] = confusion_matrix(y_test, pred)
    fitted[name] = model

summary_df = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
summary_df

## 7. Model Evaluation with Security-Focused Metrics

Print reports, plot confusion matrices, compare metrics, and identify the best model by F1 score.

In [ ]:
for name in summary_df["Model"]:
    print(f"\nClassification report: {name}")
    print(reports[name])

    cm = conf_mats[name]
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.lower().replace(' ', '_')}.png"), dpi=200)
    plt.show()

best_model_name = summary_df.iloc[0]["Model"]
best_model = fitted[best_model_name]
print("Best model by F1:", best_model_name)

metric_cols = ["Accuracy", "Precision", "Recall", "F1"]
plot_df = summary_df[["Model"] + metric_cols]

x = np.arange(len(plot_df))
width = 0.18
plt.figure(figsize=(12, 6))
for i, m in enumerate(metric_cols):
    plt.bar(x + i * width, plot_df[m], width=width, label=m)
plt.xticks(x + width * 1.5, plot_df["Model"], rotation=15)
plt.ylim(0, 1.05)
plt.title("Model Comparison: Accuracy, Precision, Recall, F1")
plt.xlabel("Model")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=200)
plt.show()

best_pred = best_model.predict(X_test_sel)
best_cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(best_cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix - Best ({best_model_name})")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_best.png"), dpi=200)
plt.show()

## 8. Decision Threshold Tuning

Tune alert threshold based on precision-recall behavior and choose an operational threshold.

In [ ]:
if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(X_test_sel)[:, 1]
else:
    scores = best_model.decision_function(X_test_sel)
    probs = 1.0 / (1.0 + np.exp(-scores))

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)
threshold_f1 = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-9, None)
best_t_idx = int(np.argmax(threshold_f1))
optimal_threshold = float(thresholds[best_t_idx]) if len(thresholds) else 0.5

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, threshold_f1, label="F1")
plt.axvline(optimal_threshold, color="red", linestyle="--", label=f"Selected={optimal_threshold:.3f}")
plt.title("Threshold Tuning Curves")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

print("Selected threshold:", round(optimal_threshold, 4))

## 9. Model Explainability for Alerts

Use permutation importance on the selected features to justify model decisions.

In [ ]:
perm = permutation_importance(best_model, X_test_sel, y_test, n_repeats=5, random_state=RANDOM_STATE)
imp_df = pd.DataFrame({
    "feature": X_test_sel.columns,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(9, 6))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="#31a354")
plt.title("Permutation Importance (Top 15)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

imp_df.head(10)

## 10. Batch and Streaming Inference Pipeline

Create reusable scoring helpers for single records, batch records, and simulated streams.

In [ ]:
def score_batch(df_rows, model, scaler, selected_feature_names, threshold=0.5):
    rows = df_rows.reindex(columns=selected_feature_names, fill_value=0)
    rows_scaled = scaler.transform(rows)
    rows_sel = pd.DataFrame(rows_scaled, columns=selected_feature_names)

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(rows_sel)[:, 1]
    else:
        scores = model.decision_function(rows_sel)
        probs = 1.0 / (1.0 + np.exp(-scores))

    labels = np.where(probs >= threshold, "THREAT DETECTED", "NORMAL")
    return pd.DataFrame({"prediction": labels, "threat_probability": probs})

sample_input = X_test_raw.iloc[0][selected_features].to_dict()
single_result = predict_threat(
    sample_input,
    best_model,
    deploy_scaler,
    selected_features,
    threshold=optimal_threshold,
)
print("Single prediction:", single_result)

batch_result = score_batch(
    X_test_raw.head(5),
    best_model,
    deploy_scaler,
    selected_features,
    threshold=optimal_threshold,
)
batch_result

## 11. Unit Tests for Detection Components

Notebook-friendly assertions for preprocessing and inference behavior.

In [ ]:
assert len(selected_features) <= 15
assert set(y.unique()).issubset({0, 1})
assert X_train_sel.shape[1] == len(selected_features)
assert deploy_scaler.n_features_in_ == len(selected_features)

result = predict_threat(
    X_test_raw.iloc[0][selected_features].to_dict(),
    best_model,
    deploy_scaler,
    selected_features,
    threshold=optimal_threshold,
)
assert "label" in result and "threat_probability" in result
assert 0.0 <= result["threat_probability"] <= 1.0
print("All notebook checks passed.")

## 12. Persisting Artifacts and Reproducible Runs

Tune the best model, save model and scaler artifacts, and print the final summary report.

In [ ]:
search_spaces = {
    "Logistic Regression": {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs", "liblinear"]},
    "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    "Decision Tree": {"max_depth": [None, 10, 20, 30], "min_samples_split": [2, 5, 10]},
    "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
    "SVM": {"C": [0.1, 1, 10], "gamma": ["scale", "auto", 0.01], "kernel": ["rbf"]},
    "LinearSVC": {"C": [0.01, 0.1, 1, 10], "loss": ["hinge", "squared_hinge"]},
}

space = search_spaces.get(best_model_name, {})
if space:
    n_comb = int(np.prod([len(v) for v in space.values()]))
    tuner = RandomizedSearchCV(
        best_model,
        param_distributions=space,
        n_iter=min(10, max(1, n_comb)),
        cv=3,
        scoring="f1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    tuner.fit(X_train_bal, y_train_bal)
    tuned_model = tuner.best_estimator_
    best_params = tuner.best_params_
else:
    tuned_model = best_model
    best_params = {}

tuned_model.fit(X_train_bal, y_train_bal)

if hasattr(tuned_model, "predict_proba"):
    tuned_probs = tuned_model.predict_proba(X_test_sel)[:, 1]
else:
    tuned_scores = tuned_model.decision_function(X_test_sel)
    tuned_probs = 1.0 / (1.0 + np.exp(-tuned_scores))

tuned_pred = (tuned_probs >= optimal_threshold).astype(int)

acc = accuracy_score(y_test, tuned_pred)
prec = precision_score(y_test, tuned_pred, zero_division=0)
rec = recall_score(y_test, tuned_pred, zero_division=0)
f1 = f1_score(y_test, tuned_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_test, tuned_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) else 0.0

joblib.dump(tuned_model, os.path.join(ARTIFACTS_DIR, "best_model.pkl"))
joblib.dump(deploy_scaler, os.path.join(ARTIFACTS_DIR, "scaler.pkl"))
joblib.dump(selected_features, os.path.join(ARTIFACTS_DIR, "feature_names.pkl"))

class_balance = y.value_counts(normalize=True).sort_index()
normal_pct = class_balance.get(0, 0.0) * 100
attack_pct = class_balance.get(1, 0.0) * 100

print("Best params:", best_params)
print("\n" + "=" * 45)
print("  ITM-390 | AI-Based Threat Detection Tool")
print("  American University of Phnom Penh")
print("=" * 45)
print(f"Dataset:         {dataset_name}")
print(f"Total Samples:   {len(df_clean)}")
print(f"Features Used:   {selected_features}")
print(f"Class Balance:   Normal={normal_pct:.2f}% | Attack={attack_pct:.2f}%")

print("\n--- Model Comparison (ranked by F1) ---")
print(f"{'Model':24} {'Acc':6} {'Prec':6} {'Rec':6} {'F1':6}")
print("-" * 55)
for _, row in summary_df.iterrows():
    marker = " <- BEST" if row['Model'] == best_model_name else ""
    print(f"{row['Model'][:24]:24} {row['Accuracy']:.3f}  {row['Precision']:.3f}  {row['Recall']:.3f}  {row['F1']:.3f}{marker}")

print(f"\nBest Model:      {best_model_name}")
print(f"Best F1-Score:   {f1:.3f}")
print(f"False Positive Rate: {fpr:.3%}")
print("Deployment:      predict_threat() function ready")
print("=" * 45)

print("\nArtifacts saved:")
print("- output/class_distribution.png")
print("- output/feature_importance.png")
print("- output/model_comparison.png")
print("- output/confusion_matrix_best.png")
print("- artifacts/best_model.pkl")
print("- artifacts/scaler.pkl")
print("- artifacts/feature_names.pkl")